In [ ]:
# 04_era1_ramp_characterization.ipynb — Era 1 (2019-2022) intra-day ramp analysis
# Re-download if runtime reset:
# !kaggle datasets download -d halcyonvector/india-power-grid-nldc-daily-psp-reports -p data --unzip

import pandas as pd
import matplotlib.pyplot as plt

hourly = pd.read_csv("data/study1_hourly.csv")
hourly["datetime"] = pd.to_datetime(hourly["datetime"], format="%Y-%m-%d %H:%M:%S")  # fixed 2026-07-11 -- study1_hourly.csv is actually ISO format, the DD-MM-YYYY assumption never matched the real data and would have crashed if run
hourly["date"] = hourly["datetime"].dt.normalize()
daily = pd.read_csv("data/study1_daily.csv", parse_dates=["date"])

# Restrict to Era 1
hourly = hourly[hourly["date"].dt.year.between(2019, 2022)].copy()
daily_era1 = daily[daily["date"].dt.year.between(2019, 2022)][["date", "share_res_pct"]]

DEMAND_COL = "National Hourly Demand"  # confirm exact column name in your CSV

# --- Hour-to-hour ramp (delta) per day ---
hourly = hourly.sort_values("datetime").reset_index(drop=True)  # sort by full datetime, not just date -- sort_values on "date" alone
# isn't guaranteed stable across the 24 same-date rows per day, which could scramble hour order
hourly["ramp"] = hourly[DEMAND_COL].diff()

# Ramp magnitude = daily max absolute hour-to-hour change
daily_ramp = hourly.groupby(hourly["date"].dt.date)["ramp"].apply(lambda x: x.abs().max())
daily_ramp = daily_ramp.reset_index()
daily_ramp.columns = ["date", "ramp_magnitude"]
daily_ramp["date"] = pd.to_datetime(daily_ramp["date"])

# Ramp frequency = count of hours where |delta| exceeds a threshold (e.g. top 10% of ramps)
threshold = hourly["ramp"].abs().quantile(0.9)
daily_freq = hourly.groupby(hourly["date"].dt.date)["ramp"].apply(lambda x: (x.abs() > threshold).sum())
daily_freq = daily_freq.reset_index()
daily_freq.columns = ["date", "ramp_frequency"]
daily_freq["date"] = pd.to_datetime(daily_freq["date"])

# --- Merge with RES share ---
merged = daily_ramp.merge(daily_freq, on="date").merge(daily_era1, on="date", how="left")

# --- Monthly aggregation for trend clarity ---
merged["month"] = merged["date"].dt.to_period("M")
monthly = merged.groupby("month").agg(
    ramp_magnitude=("ramp_magnitude", "mean"),
    ramp_frequency=("ramp_frequency", "mean"),
    share_res_pct=("share_res_pct", "mean"),
).reset_index()
monthly["month"] = monthly["month"].dt.to_timestamp()

# --- The key chart: ramp magnitude trend vs RES share ---
# Fixed 2026-07-11: was a dual y-axis chart (two independent scales sharing one plot,
# the classic misleading-chart pattern -- the two lines' apparent crossing points and
# relative positions don't mean anything since each has its own arbitrary scale).
# Replaced with both series indexed to a common 0-100 scale (min-max per series) on a
# single axis -- the co-trend is still directly comparable, but now honestly so.
def index_0_100(s):
    return (s - s.min()) / (s.max() - s.min()) * 100

plt.figure(figsize=(12, 5))
plt.plot(monthly["month"], index_0_100(monthly["ramp_magnitude"]), color="#2a78d6", label="Ramp magnitude (indexed)")
plt.plot(monthly["month"], index_0_100(monthly["share_res_pct"]), color="#e87ba4", label="RES share % (indexed)")
plt.ylabel("Indexed to 0-100 (each series' own min-max)")
plt.legend()
plt.title("Era 1 (2019-2022): Intra-day ramp magnitude vs RES share (indexed to a common scale)")
plt.show()

# --- Correlation check -
print(monthly[["ramp_magnitude", "ramp_frequency", "share_res_pct"]].corr())

monthly.to_csv("data/era1_ramp_vs_res_share.csv", index=False)
